# Custom CNN architecture

Build and check the four-class CNN. This notebook does not train the model. It imports make_dataset from the shared src/data_pipeline.py script; keep the src folder in the repository. Uses the fixed ResNet50 shared split. Train a fresh model on this split.

In VS Code, select the repository `.venv` Python kernel and run cells from top to bottom. The shared src/data_pipeline.py helper is retained for model.ipynb.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dataset").is_dir() and (p / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the project repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)


In [ ]:
import tensorflow as tf



In [ ]:
from tensorflow.keras import layers, models



In [ ]:
SEED = 42



In [ ]:
def build_model():
    inputs = layers.Input(shape=(224, 224, 3), name="image")

    # Input images are already normalized by data_pipeline.py.
    # RandomRotation uses fractions of a full turn: 15 / 360.
    x = layers.RandomRotation(
        factor=15 / 360,
        fill_mode="reflect",
        seed=SEED,
    )(inputs)
    x = layers.RandomZoom(
        height_factor=0.1,
        width_factor=0.1,
        fill_mode="reflect",
        seed=SEED + 1,
    )(x)
    x = layers.RandomFlip(
        mode="horizontal",
        seed=SEED + 2,
    )(x)

    # Blocks 1–3: two convolutions per block.
    for block, filters, dropout in [
        (1, 32, 0.25),
        (2, 64, 0.25),
        (3, 128, 0.30),
    ]:
        x = layers.Conv2D(
            filters, 3,
            padding="same",
            activation="relu",
            name=f"block{block}_conv1",
        )(x)
        x = layers.BatchNormalization(
            name=f"block{block}_bn",
        )(x)
        x = layers.Conv2D(
            filters, 3,
            padding="same",
            activation="relu",
            name=f"block{block}_conv2",
        )(x)
        x = layers.MaxPooling2D(
            pool_size=2,
            name=f"block{block}_pool",
        )(x)
        x = layers.Dropout(
            dropout,
            name=f"block{block}_dropout",
        )(x)

    # Block 4: one convolution.
    x = layers.Conv2D(
        256, 3,
        padding="same",
        activation="relu",
        name="block4_conv",
    )(x)
    x = layers.BatchNormalization(name="block4_bn")(x)
    x = layers.MaxPooling2D(pool_size=2, name="block4_pool")(x)
    x = layers.Dropout(0.30, name="block4_dropout")(x)

    x = layers.GlobalAveragePooling2D(name="global_pool")(x)
    x = layers.Dense(256, activation="relu", name="dense")(x)
    x = layers.Dropout(0.50, name="dense_dropout")(x)
    outputs = layers.Dense(
        4,
        activation="softmax",
        name="class_probabilities",
    )(x)

    model = models.Model(inputs, outputs, name="custom_cnn")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )

    return model



## Run the check


In [ ]:
from src.data_pipeline import make_dataset

tf.keras.utils.set_random_seed(SEED)
model = build_model()
model.summary()

# Verify inference using two training images.
images, labels = next(iter(make_dataset("train")))
probabilities = model(images[:2], training=False)

tf.debugging.assert_shapes([(probabilities, (2, 4))])
tf.debugging.assert_all_finite(
    probabilities, "Model produced invalid probabilities"
)
tf.debugging.assert_near(
    tf.reduce_sum(probabilities, axis=1),
    tf.ones(2),
    atol=1e-5,
)

print("\nOutput shape:", probabilities.shape)
print("Probability sums:", tf.reduce_sum(probabilities, axis=1).numpy())
print("Model check completed. Training has not started.")